# Capstone #1: Research Agent

*Notebook 11*

Put it all together.

Combine web search, file search, and Code Interpreter into one agent.

It researches a topic, analyzes documents, and produces a structured report.


---

## 🔧 Setup

In [ ]:
import os
import re
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(dotenv_path=Path("..") / ".env")

from openai import OpenAI
from agents import Agent, CodeInterpreterTool, FileSearchTool, Runner, ToolCallItem, WebSearchTool

MODEL = "gpt-5-mini"
REASONING_MODEL = "gpt-5"

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def called_tool_types(result) -> set[str]:
    """The set of hosted tool-call types the run actually invoked."""
    return {
        getattr(item.raw_item, "type", "")
        for item in result.new_items
        if isinstance(item, ToolCallItem)
    }


print(f"✅ Setup complete: Using {MODEL}")

---

## 🏗️ System Architecture

The research agent combines all three built-in tools from Lessons 08-10:

- **Web Search** (Lesson 08): finds current information and news

- **File Search** (Lesson 09): queries private reference documents

- **Code Interpreter** (Lesson 10): computes and analyzes quantitative data

**Workflow:** Topic → agent autonomously chooses tools → structured report.

---

## 📄 Phase 1: Build the Knowledge Base

We'll create an internal reference document and upload it to a vector store.

This gives the agent access to private context that web search won't find.

In [ ]:
# Internal reference document (synthetic demo data): private context the web doesn't have
reference_doc = """# AI Adoption Survey (Synthetic Demo Data): Internal Research Summary

## Survey Overview
Fictional survey created for this course: 847 respondents across technology, healthcare,
finance, and retail sectors. Companies ranged from 50 to 10,000+ employees.

## Key Findings

### Adoption Rates
- 73% of companies surveyed are actively using AI tools in production
- 18% are in pilot or evaluation phase
- 9% have no AI initiatives currently

### Primary Use Cases (ranked by adoption)
1. Customer support automation: 61%
2. Data analysis and reporting: 58%
3. Content generation: 47%
4. Code assistance: 44%
5. Document processing: 39%

### Biggest Barriers to Adoption
- Data privacy and security concerns: 67%
- Integration with existing systems: 54%
- Lack of skilled staff: 48%
- Cost and ROI uncertainty: 43%

### Budget Trends
- Average AI budget increase year-over-year: 34%
- Companies with dedicated AI teams: 41%
- Expected to increase AI investment next year: 78%

### Satisfaction
- Exceeded expectations: 29%
- Met expectations: 51%
- Below expectations: 20%
"""

# Save locally
doc_path = Path("ai_adoption_survey.txt")
doc_path.write_text(reference_doc)

# --------------------------------------------------------------
print(f"✅ Reference document created: {doc_path}")

#### Upload to Vector Store

In [ ]:
print("Uploading to vector store...\n")

vector_store = client.vector_stores.create(name="Research Agent Knowledge Base")

with open(doc_path, "rb") as file_handle:
    file_batch = client.vector_stores.file_batches.upload_and_poll(
        vector_store_id=vector_store.id,
        files=[file_handle]
    )

if file_batch.file_counts.failed:
    raise RuntimeError(f"File processing failed: {file_batch.file_counts.failed} file(s)")

# --------------------------------------------------------------
print(f"✅ Vector store ready: {vector_store.id}")
print(f"   Files: {file_batch.file_counts.completed} processed")

With the knowledge base ready, we can build one agent.

It uses both public web data and our internal survey.

## 🤖 Phase 2: Build the Research Agent

All three tools live in one agent, and its instructions tell it how to use each.

Pattern to copy:

- List every tool the agent might need

- Give numbered instructions for when to use each

- Start with default parameters

In your own project, swap the vector store contents, instructions, and `ResearchReport` fields.

The report format is deliberately constrained: one-sentence findings, summaries instead of pasted text, and citations only in the source list.

Hosted-tool output remains model-generated. Validate important source facts and citation metadata before relying on the report.

In [ ]:
from pydantic import BaseModel
from typing import List

class ResearchReport(BaseModel):
    executive_summary: str
    key_findings: List[str]
    data_analysis: str
    sources: List[str]

research_instructions = (
    "You are a thorough research analyst. When given a research topic:\n\n"
    "1. Use web search to find current information, news, and trends\n"
    "2. Use file search to find relevant internal data and context\n"
    "3. Use the Python tool to compute statistics or analyze any numerical data\n"
    "The internal survey is synthetic demo data: label it that way in the report.\n"
    "Preserve source numbers exactly: never merge, alter, or invent them.\n"
    "Keep the executive summary to 2-3 sentences and each key finding to ONE sentence.\n"
    "Summarize documents in your own words: never paste document text verbatim.\n"
    "Cite web sources only in the sources list, never inline in the summary, findings, or analysis.\n"
    "Keep source entries brief: the displayed list is rebuilt from the run's citation records.\n"
    "4. Synthesize all findings into a structured report with these sections:\n"
    "    - Executive Summary (2-3 sentences)\n"
    "    - Key Findings (bullet points)\n"
    "    - Data Analysis (numbers and statistics)\n"
    "    - Sources\n\n"
    "Always cite your sources. Be specific with numbers."
)

research_agent = Agent(
    name="ResearchAgent",
    instructions=research_instructions,
    model=MODEL,
    output_type=ResearchReport,
    tools=[
        WebSearchTool(),
        FileSearchTool(
            vector_store_ids=[vector_store.id],
            max_num_results=3
        ),
        CodeInterpreterTool(tool_config={
            "type": "code_interpreter",
            # Auto mode creates a container or reuses one already in model context.
            "container": {"type": "auto"}
        })
    ]
)

# --------------------------------------------------------------
print("✅ Research agent ready")
print("    Tools: WebSearch + FileSearch + CodeInterpreter")

---

## 🚀 Phase 3: Run the Research Agent

The agent can use web search, file search, and Code Interpreter in one run.

⚠️ **Security note:** Tool output (web results, doc chunks) is untrusted.

Validate before passing it downstream (more in Lesson 21).

In [ ]:
research_topic = (
    "Research the current state of enterprise AI adoption: trends, barriers, and outlook. "
    "Use the internal survey and current web sources. "
    "Use the Python tool to estimate how many of the 847 respondents represent the 73% adoption rate."
)

print("🔬 RESEARCH AGENT RUNNING")
print("=" * 60)
print(f"Topic: {research_topic}")
print("\nResearching...\n")

# TODO (pre-recording): re-verify a clean run. On 2026-07-20 the Responses
# API intermittently returned corrupted text on web-search runs (server-side,
# request IDs in scratchpad evidence bundle). The gates below catch it.
# Multi-tool runs can fail at any step, so surface failures clearly
try:
    research_result = await Runner.run(research_agent, input=research_topic)
except Exception as error:
    print(f"Research failed: {error}")
    research_result = None

if research_result:
    report = research_result.final_output

    print("=" * 60)
    print("📋 RESEARCH REPORT")
    print("=" * 60)
    print(f"\nExecutive Summary:\n{report.executive_summary}\n")
    print("Key Findings:")
    for finding in report.key_findings:
        print(f"  • {finding}")
    print(f"\nData Analysis:\n{report.data_analysis}\n")
    print("Sources:")
    for source in report.sources:
        print(f"  • {source}")
    print("=" * 60)

    # Because we used output_type=ResearchReport, `report` is a validated Python object
    # with .executive_summary, .key_findings, etc. It has the expected shape for
    # downstream code, but facts and sources still need review.

    # Evidence: which of the three tools actually ran
    called = called_tool_types(research_result)
    expected_tools = {
        "Web Search": "web_search_call",
        "File Search": "file_search_call",
        "Code Interpreter": "code_interpreter_call",
    }
    print("\nTool use:")
    for label, call_type in expected_tools.items():
        print(f"  {label}: {'yes' if call_type in called else 'no'}")

    # Gate 1: survey facts must survive as PAIRS in the substantive fields.
    # (Global number presence proved inadequate: a corrupted report once
    #  paired a use case with the wrong percentage and still contained
    #  every number somewhere.)
    fields_text = " ".join(
        [report.executive_summary] + report.key_findings + [report.data_analysis]
    ).lower()

    def near(a: str, b: str, window: int = 60) -> bool:
        forward = re.escape(a.lower()) + ".{0," + str(window) + "}" + re.escape(b.lower())
        reverse = re.escape(b.lower()) + ".{0," + str(window) + "}" + re.escape(a.lower())
        return bool(re.search(forward, fields_text, re.S) or re.search(reverse, fields_text, re.S))

    required_checks = {
        "labeled synthetic": "synthetic" in fields_text,
        "73% paired with 847": near("73%", "847"),
        "618 estimate present": "618" in fields_text,
        "privacy paired with 67%": near("privacy", "67%"),
    }

    # Survey pairs the report may mention: if a phrase appears, its number must too
    conditional_pairs = [
        ("customer support automation", "61%"),
        ("data analysis", "58%"),
        ("content generation", "47%"),
        ("code assistance", "44%"),
        ("document processing", "39%"),
        ("skilled staff", "48%"),
    ]
    conditional_results = {}
    for phrase, pct in conditional_pairs:
        if phrase in fields_text:
            conditional_results[f"{phrase} = {pct}"] = near(phrase, pct)

    # Gate 2: no inline citation markup in the substantive fields
    no_inline_markup = "](" not in fields_text and "http" not in fields_text

    # Gate 3: citation annotations must start on a clean text boundary
    boundary_issues = 0
    annotation_count = 0
    web_sources = []
    for item in research_result.new_items:
        if item.type != "message_output_item":
            continue
        for part in item.raw_item.content:
            part_text = getattr(part, "text", "")
            for annotation in (getattr(part, "annotations", None) or []):
                start = getattr(annotation, "start_index", None)
                if start is not None:
                    annotation_count += 1
                    if start > 0 and part_text[start - 1].isalnum():
                        boundary_issues += 1
                url = getattr(annotation, "url", None)
                if url:
                    title = getattr(annotation, "title", "") or url
                    entry = f"{title} | {url.split('?')[0]}"
                    if entry not in web_sources:
                        web_sources.append(entry)

    # Canonical sources: annotation metadata is reliable even when text is not
    canonical_sources = ["Internal survey (synthetic demo data) | ai_adoption_survey.txt"] + web_sources
    discarded = len(report.sources)
    report.sources = canonical_sources

    print("\nSources (assembled from citation records):")
    for source in report.sources:
        print(f"  • {source[:90]}")
    print(f"  (model-written source entries discarded: {discarded})")

    print("\nSource-fidelity gates:")
    for label, ok in required_checks.items():
        print(f"  {label}: {'✅' if ok else '❌'}")
    for label, ok in conditional_results.items():
        print(f"  {label} (mentioned): {'✅' if ok else '❌'}")
    print(f"  no inline citation markup in report fields: {'✅' if no_inline_markup else '❌'}")
    print(f"  citation annotations on clean boundaries: "
          f"{'✅' if boundary_issues == 0 else '❌'} ({annotation_count - boundary_issues}/{annotation_count})")
    print(f"  web sources recovered from citation records: {'✅' if len(web_sources) >= 1 else '❌'}")

    all_green = (
        all(required_checks.values())
        and all(conditional_results.values())
        and no_inline_markup
        and boundary_issues == 0
        and len(web_sources) >= 1
    )
    if not all_green:
        print("\n⚠️  Report failed source-fidelity validation. Do not rely on this output.")


## 🧹 Demo Cleanup

Removes the demo's sample files so the exercises start clean.

In [ ]:
# Clean up local sample files before exercises
for file in ["ai_adoption_survey.txt"]:
    path = Path(file)
    if path.exists():
        path.unlink()
        print(f"✅ Local file deleted: {file}")

---

## 💪 Practice Exercises

### Exercise 1: Export Research Report to Markdown

*Covers: saving agent output, writing results to a file*

Save the research report to a `.md` file using the existing `research_result`.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 1: Export Research Report to Markdown
# --------------------------------------------------------------
# Objective: Save the existing research report to a markdown file.

from datetime import datetime

# TODO 1: Create a markdown header with the topic and current date
#          Hint: date_str = datetime.now().strftime("%Y-%m-%d")
#                header = f"# Research Report: {research_topic}\nDate: {date_str}\n\n"

# TODO 2: Build the report body from all four ResearchReport fields:
#          executive_summary, key_findings, data_analysis, sources
#          (research_result.final_output.executive_summary, and so on)

# TODO 3: Write the combined content to "report.md" using Path("report.md").write_text()

# TODO 4: Print a confirmation message showing the file path

# TODO 5: (Optional) Delete report.md when you're done to keep the folder clean

# --- Write your code below this line ---

---

### Exercise 2: Evaluate One Research Report

*Covers: judge-agent evaluation, scoring a structured report*

Apply the judge-agent evaluation pattern from Lesson 07 to one research report.

Focus on the judge step, not a full test harness.

Good golden tests for this kind of agent check two things:

- **Content coverage**: did it include required facts?

- **Tool-backed grounding**: did it cite sources or use internal data when expected?

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 2: Evaluate One Research Report
# --------------------------------------------------------------
# Objective: Score one research report using a judge agent (Lesson 07 pattern).

from pydantic import BaseModel, Field

# A golden test: the topic we already ran, with criteria the report should meet
golden_test = {
    "topic": "Current state of AI adoption in enterprise",
    "must_contain": ["73%", "privacy"],  # facts from the internal survey
}

# Define an EvalResult model for the judge agent
# (no passed field: pass/fail comes from the deterministic checks below,
#  the judge only scores quality)
class EvalResult(BaseModel):
    score: int = Field(ge=0, le=10)
    feedback: str

# TODO 1: Check the golden_test criteria deterministically in Python:
#          confirm each must_contain string appears in
#          report.model_dump_json()

# TODO 2: Create a judge agent using REASONING_MODEL and output_type=EvalResult
#          Instructions: score the report's clarity, grounding, and usefulness

# TODO 3: Build the judge input from golden_test and
#          report.model_dump_json(). Judge the captured report.
#          Do not rerun the research agent.

# TODO 4: Check tool use deterministically:
#          required = {"web_search_call", "file_search_call", "code_interpreter_call"}
#          Confirm required <= called_tool_types(research_result)

# TODO 5: Derive the final verdict in Python:
#          all deterministic checks pass AND the judge score >= 7

# --- Write your code below this line ---

---

## 🎯 Key Takeaways

**One agent can choose among multiple tools:**

- Web search for current information

- File search for private context

- Code Interpreter for computation

- Inspect run items to prove which tools actually ran
<br>
<br>

**Instructions guide tool use:**

- Explicit instructions about when to use each tool produce more consistent results

- Structured output (`output_type=ResearchReport`) guarantees the report's shape on a successful run, not its content

- Handle errors where the caller can recover: a multi-tool run can fail at any step
<br>
<br>

**Files and vector stores are separate resources:**

- Delete uploaded files with `client.files.delete(...)` and the store with `client.vector_stores.delete(...)`

- In a real app, keep the store and reuse its ID until the documents change

---

## 📍 Next Step

**Notebook 12: Handoffs**  

Learn how to route tasks between specialized agents using the handoff pattern.

---

##### 🔧 [Troubleshooting Guide](https://github.com/barrettscott/openai-agents/blob/main/TROUBLESHOOTING.md#lesson-11-capstone-1--research-agent)

---

#### Cleanup: Delete Uploaded Files and Vector Store

Uploaded files and vector stores persist on OpenAI's servers until deleted.

Deleting the store does not delete the files you uploaded, so remove both.

Run this cell when you're done to avoid ongoing storage charges.

In [ ]:
# Vector stores and uploaded Files are separate objects: delete both
uploaded_file_ids = [
    store_file.id
    for store_file in client.vector_stores.files.list(
        vector_store_id=vector_store.id
    ).data
]

for file_id in uploaded_file_ids:
    client.files.delete(file_id)

client.vector_stores.delete(vector_store.id)

print(f"✅ Uploaded files deleted: {len(uploaded_file_ids)}")
print(f"✅ Vector store deleted: {vector_store.id}")

---